# Dev Notebook for One-Tower Modeling

In [54]:
import random
import torch
import io
import pyarrow as pa
import os
import copy
from sacred import Experiment
from PIL import Image
from tqdm import tqdm
import numpy as np
import skimage.io as skio
import matplotlib.pyplot as plt
from refer import REFER

from meter.modules.heads import Pooler

from torch.utils.data import DataLoader
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning import LightningDataModule

from torch.optim import AdamW

from transformers import AutoConfig
from transformers import ElectraTokenizer, ViltFeatureExtractor
from transformers import AutoProcessor, AutoImageProcessor, AutoTokenizer
from transformers import AutoModel, AutoModelForSequenceClassification

from refcoco_utils import get_bounded_subimage
from refcoco_utils import _config
from refcoco_utils import _loss_names

from meter.transforms import keys_to_transforms
from meter.config import ex
from meter.modules import METERTransformerSS
from meter.datamodules.multitask_datamodule import MTDataModule
from meter.datasets.base_dataset import BaseDataset

In [2]:
refer_root = "/home/claytonfields/nlp/code/data/coco"

In [4]:
_config = {  
    "exp_name":"finetune_mrpc",
    "seed" : 42,
    # "datasets" : ["coco", "vg", "sbu", "gcc"],
    # "datasets" : ["coco", "vg"],
    "datasets" : ["coco"],
    "loss_names" :{'itm': 0,
    'mlm': 0,
    'mpp': 0,
    'vqa': 0,
    'vcr': 0,
    'vcr_qar': 0,
    'nlvr2': 0,
    'irtr': 0,
    'contras': 0,
    'snli': 0,
    'ref': 0,
    'mrpc':1
    },
    "batch_size" : 32,  # this is a desired batch size; pl trainer will accumulate gradients when per step batch is smaller.

    # Image setting
    "image_encoder" : "facebook/deit-tiny-patch16-224",
    "random_init_vision_encoder" : False,
    "image_encoder_hidden_size" : 192,
    "image_size" : 224,
    "patch_size" : 16,
    "draw_false_image" : 1,
    "image_only" : False,
    "resolution_before" : 224,
    "train_transform_keys" : ["imagenet"],
    "val_transform_keys" : ["imagenet"],

    # Text Setting
    "text_encoder" : "google/electra-small-discriminator",
    "random_init_text_encoder" : False,
    "text_encoder_hidden_size" : 256,
    "vocab_size" : 30522,
    "whole_word_masking" : False, # note that whole_word_masking does not work for RoBERTa
    "mlm_prob" : 0.15,
    "draw_false_text" : 0,
    "vqav2_label_size" : 3129,
    "max_text_len" : 128,

    # Architecture Setting
    "two_tower" : True,
    "multi_model_encoder" : 'dandelin/vilt-b32-mlm',

    # CrossLayer Setting
    "num_cross_layers" : 6,
    "cross_layer_hidden_size" : 256,
    "num_cross_layer_heads" : 4,
    "cross_layer_mlp_ratio" : 4,
    "cross_layer_drop_rate" : 0.1,
    
    # Optimizer Setting
    "optim_type" : "adamw",
    "learning_rate" : 5e-5,
    "weight_decay" : 0.0,
    "decay_power" : 1,
    "max_epoch" : 3,
    "max_steps" : 100000,
    "warmup_steps" : 0,
    "end_lr" : 0,
    "lr_mult_head" : 5,  # multiply lr for downstream heads
    "lr_mult_cross_modal" : 5,  # multiply lr for the cross-modal module

    # Encoder Settings
    "freeze_image_encoder" : True,
    "freeze_text_encoder" : False,
    'freeze_cross_modal_layers' : True,
    
    'text_only' : True,
    

    # Downstream Setting
    "get_recall_metric" : False,
    
    'freeze' : True,
    
    "model_type" : "METER",

    # PL Trainer Setting
    "resume_from" : None,
    "fast_dev_run" : False,
    "val_check_interval" : 1.0,
    "test_only" : False,

    "data_root" : "/home/claytonfields/nlp/code/meter/data/arrow",
    "log_dir" : "result",
    "per_gpu_batchsize" : 32,  # you should define this manually with per_gpu_batch_size:#
    "num_gpus" : 1,
    "num_nodes" : 1,
    "load_path" : "/home/claytonfields/nlp/code/meter/result/mlm_itm_seed0_from_/meter_electra_small_deit_tiny_p16_is224_bs288_is1M/checkpoints/epoch=43-step=898039.ckpt",
    # "load_path" : '/home/claytonfields/nlp/code/meter/result/mlm_itm_deit_fr_electra_fr_is224_ps16_bs336_pgbs84_ts100k/checkpoints/epoch=5-step=96215.ckpt',
    "num_workers" : 12,
    "precision" : 32
}

In [5]:
# model = METERTransformerSS(_config)

In [6]:
vilt_config = AutoConfig.from_pretrained('dandelin/vilt-b32-mlm-itm')
vilt = AutoModel.from_pretrained('dandelin/vilt-b32-mlm-itm')
vilt

ViltModel(
  (embeddings): ViltEmbeddings(
    (text_embeddings): TextEmbeddings(
      (word_embeddings): Embedding(30522, 768)
      (position_embeddings): Embedding(40, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (patch_embeddings): ViltPatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32))
    )
    (token_type_embeddings): Embedding(2, 768)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): ViltEncoder(
    (layer): ModuleList(
      (0-11): 12 x ViltLayer(
        (attention): ViltAttention(
          (attention): ViltSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=Fa

In [7]:
vilt.num_parameters()

111595008

In [9]:
dm = MTDataModule(_config, dist=False)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'ElectraTokenizer'. 
The class this function is called from is 'BertTokenizer'.


In [11]:
dm.prepare_data()
dm.setup('train')

/home/claytonfields/nlp/code/meter/meter/datasets/coco_caption_karpathy_dataset.py:18: FutureWarning: promote has been superseded by mode='default'.
  super().__init__(*args, **kwargs, names=names, text_column_name="caption")


In [37]:
data = dm.train_dataset[1]

In [38]:
text = data['text'][0]
text

'A woman holding a cake with her left hand.'

In [70]:
image = data['image'][0]
image = image.unsqueeze(0)
image

tensor([[[[-1.4158, -1.3130, -1.3644,  ...,  1.8037,  1.7865,  1.7352],
          [-1.3302, -1.3130, -1.3815,  ...,  1.8893,  1.8893,  1.8037],
          [-1.2445, -1.2959, -1.2788,  ...,  1.9578,  1.9578,  1.8379],
          ...,
          [-1.8610, -1.9124, -1.9295,  ...,  1.6838,  1.6667,  1.7180],
          [-1.9295, -1.8610, -1.8953,  ...,  1.6324,  1.7180,  1.7865],
          [-1.9295, -1.8953, -1.8782,  ...,  1.7009,  1.7865,  1.8037]],

         [[-1.3880, -1.2829, -1.3179,  ...,  2.1134,  2.0959,  1.9909],
          [-1.3179, -1.2829, -1.3354,  ...,  2.2010,  2.2010,  2.0434],
          [-1.2129, -1.2829, -1.2654,  ...,  2.2710,  2.2710,  2.0784],
          ...,
          [-1.6155, -1.6856, -1.6856,  ...,  1.5007,  1.5707,  1.6583],
          [-1.6856, -1.6155, -1.7031,  ...,  1.5007,  1.6758,  1.7633],
          [-1.7031, -1.6331, -1.7206,  ...,  1.6057,  1.7808,  1.7983]],

         [[-1.2641, -1.1944, -1.2119,  ...,  2.1694,  2.1171,  2.0300],
          [-1.1944, -1.1770, -

In [74]:
data_root = '/home/claytonfields/nlp/code/data/coco'  # contains refclef, refcoco, refcoco+, refcocog and images
dataset = 'refcoco' 
splitBy = 'unc'
refer = REFER(data_root, dataset, splitBy)

loading dataset refcoco into memory...
testing
creating index...
index created.
DONE (t=7.82s)


In [75]:
refer.Imgs

{98304: {'license': 1,
  'file_name': 'COCO_train2014_000000098304.jpg',
  'coco_url': 'http://mscoco.org/images/98304',
  'height': 424,
  'width': 640,
  'date_captured': '2013-11-21 23:06:41',
  'flickr_url': 'http://farm6.staticflickr.com/5062/5896644212_a326e96ea9_z.jpg',
  'id': 98304},
 52461: {'license': 3,
  'file_name': 'COCO_train2014_000000052461.jpg',
  'coco_url': 'http://mscoco.org/images/52461',
  'height': 484,
  'width': 640,
  'date_captured': '2013-11-17 10:55:11',
  'flickr_url': 'http://farm9.staticflickr.com/8157/7574894824_3cdc8163d9_z.jpg',
  'id': 52461},
 131074: {'license': 1,
  'file_name': 'COCO_train2014_000000131074.jpg',
  'coco_url': 'http://mscoco.org/images/131074',
  'height': 428,
  'width': 640,
  'date_captured': '2013-11-21 01:03:06',
  'flickr_url': 'http://farm9.staticflickr.com/8308/7908210548_33e532d119_z.jpg',
  'id': 131074},
 524291: {'license': 3,
  'file_name': 'COCO_train2014_000000524291.jpg',
  'coco_url': 'http://mscoco.org/images/5

In [78]:
# file_name = 'COCO_train2014_000000173056_1.jpg'
img_id = 98304
img = refer.Imgs[img_id]
I = skio.imread(os.path.join(refer.IMAGE_DIR, img['file_name']))
I

array([[[152, 157, 160],
        [151, 156, 159],
        [150, 155, 158],
        ...,
        [125, 123, 111],
        [125, 123, 111],
        [125, 123, 111]],

       [[150, 155, 159],
        [151, 156, 160],
        [152, 157, 161],
        ...,
        [126, 124, 112],
        [125, 123, 111],
        [125, 123, 111]],

       [[150, 155, 159],
        [151, 156, 160],
        [153, 158, 162],
        ...,
        [126, 122, 111],
        [125, 121, 110],
        [125, 121, 110]],

       ...,

       [[153, 156, 163],
        [153, 156, 163],
        [153, 156, 163],
        ...,
        [128, 122,  96],
        [129, 122,  96],
        [128, 121,  95]],

       [[151, 156, 162],
        [152, 157, 163],
        [152, 157, 163],
        ...,
        [127, 121,  97],
        [127, 121,  97],
        [127, 121,  97]],

       [[152, 157, 163],
        [152, 157, 163],
        [152, 157, 163],
        ...,
        [126, 120,  96],
        [126, 120,  96],
        [127, 121,  97]]

In [65]:
from transformers import SwinModel
swin = SwinModel.from_pretrained("microsoft/swin-tiny-patch4-window7-224")
swin(image.unsqueeze(0))

SwinModelOutput(last_hidden_state=tensor([[[ 0.7206,  0.3245, -1.9140,  ...,  0.6992,  0.1120,  1.2279],
         [ 1.2395,  0.4779, -2.8043,  ...,  0.1212, -0.2265,  1.0833],
         [ 1.9992, -0.3482, -2.7675,  ...,  0.5500, -0.2383,  1.9215],
         ...,
         [ 4.2901,  1.8620,  2.1743,  ...,  0.0670,  4.3576, -3.6659],
         [ 0.1727,  0.8066, -0.2098,  ...,  1.0178,  2.1442, -2.4030],
         [ 0.6652, -0.5127, -0.2259,  ...,  0.5176, -0.2466,  0.2434]]],
       grad_fn=<NativeLayerNormBackward0>), pooler_output=tensor([[ 1.4630e+00,  4.0268e-01, -8.6956e-01, -5.4830e-01, -1.4620e-01,
         -3.1436e-01,  2.1247e-01,  1.2729e+00,  2.9421e-01, -2.6679e-01,
          1.0392e+00,  7.1749e-01, -9.3560e-01, -2.9883e-01, -3.1078e-01,
         -8.4082e-01, -1.2354e+00, -6.7206e-01, -1.9497e-01, -6.9144e-01,
          3.7141e-01,  5.7014e-02, -4.8522e-01, -5.6672e-01, -1.0114e+00,
          5.2930e-02,  7.9692e-01, -7.6103e-01, -1.0248e+00,  8.5748e-01,
         -2.0904e+00, 

In [83]:
inputs = preprocessor(images=I, return_tensors="pt")
inputs

{'pixel_values': tensor([[[[ 0.0431,  0.0902,  0.1294,  ...,  0.2157,  0.2235,  0.2235],
          [ 0.0745,  0.0980,  0.1294,  ...,  0.2078,  0.2235,  0.2235],
          [ 0.0588,  0.1059,  0.1373,  ...,  0.2078,  0.2157,  0.2314],
          ...,
          [-0.7804, -0.8196, -0.8353,  ..., -0.0039, -0.0039, -0.0039],
          [-0.8196, -0.8196, -0.8353,  ...,  0.0039,  0.0039, -0.0039],
          [-0.8196, -0.8196, -0.8353,  ...,  0.0118,  0.0039, -0.0039]],

         [[ 0.0196,  0.0745,  0.1137,  ...,  0.1529,  0.1608,  0.1608],
          [ 0.0196,  0.0667,  0.1216,  ...,  0.1529,  0.1686,  0.1608],
          [ 0.0275,  0.0745,  0.1216,  ...,  0.1608,  0.1765,  0.1686],
          ...,
          [-0.7804, -0.8353, -0.8431,  ..., -0.0745, -0.0745, -0.0824],
          [-0.8275, -0.8353, -0.8431,  ..., -0.0667, -0.0745, -0.0667],
          [-0.8353, -0.8353, -0.8431,  ..., -0.0588, -0.0745, -0.0588]],

         [[-0.0667,  0.0039,  0.0745,  ...,  0.0510,  0.0588,  0.0588],
          [-0

In [86]:
preprocessor = AutoImageProcessor.from_pretrained("google/mobilenet_v2_1.4_224")
model = AutoModel.from_pretrained("google/mobilenet_v2_1.4_224")

inputs = preprocessor(images=I, return_tensors="pt")

outputs = model(image)
outputs

BaseModelOutputWithPoolingAndNoAttention(last_hidden_state=tensor([[[[0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 1.7066e-01,
           0.0000e+00, 0.0000e+00],
          [0.0000e+00, 6.6093e-01, 0.0000e+00,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          ...,
          [1.6495e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [0.0000e+00, 0.0000e+00, 4.1370e-02,  ..., 0.0000e+00,
           2.3750e-01, 0.0000e+00],
          [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 1.5096e+00,
           1.7121e+00, 0.0000e+00]],

         [[0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [0.0000e+00, 2.4432e+00, 2.4607e-01,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [0.0000e+00, 2.5222e+00, 0.0000e+00,  ..., 0.0000e+00,
           2.2675e+00, 0.0000e+00],
          ...,


In [64]:
vilt(image)

ValueError: too many values to unpack (expected 2)

### Classification Head

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset
from torchtext.datasets import mrpc

In [ ]:
data_dict = load_dataset('glue', 'mrpc', split='train').to_dict()

In [ ]:
data_dict.keys()

In [ ]:
data_dict['idx'].__len__()#[-1]

In [ ]:
class GlueDataset(torch.utils.data.Dataset):
    def __init__(self, task, split, tokenizer):

        self.task = task
        self.split = split
        self.tokenizer = tokenizer

        self.data_dict = load_dataset('glue', self.task, split=self.split).to_dict()
        self.sentence1 = self.data_dict['sentence1']
        self.sentence2 = self.data_dict['sentence2']
        self.label = self.data_dict['label']
        self.idx = self.data_dict['idx']

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, index):

        sent1 = self.sentence1[index]
        sent2 = self.sentence2[index]
        label = self.label[index]
        # idx = self.idx[index]

        ret = tokenizer(
            sent1, 
            sent2,
            max_length=60,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        ret = {k: v.squeeze() for k,v in ret.items()}
        ret['labels'] = label
        return ret

    

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model.text_encoder.config._name_or_path)
tokenizer

In [ ]:
ds = GlueDataset('mrpc', 'train', tokenizer)
ds

In [ ]:
dl = DataLoader(ds, batch_size=10)
dl

In [ ]:
text_pooler = Pooler(_config['text_encoder_hidden_size'])
loss_fn = F.cross_entropy

In [ ]:
# Objective
batch = next(iter(dl))
labels = labels = batch.pop('label',None)
hidden_state = model.text_encoder(**batch).last_hidden_state#.squeeze()
cls_feat = text_pooler(hidden_state)
# cls_feat.shape
logits = model.mrpc_classifier(cls_feat)
loss = loss_fn(logits, labels)
loss

In [ ]:
class GlueDataModule(LightningDataModule):
    def __init__(self, config, task, batch_size):
        super().__init__()

        self.task = task
        self.batch_size = batch_size
        self.tokenizer = AutoTokenizer.from_pretrained(config['text_encoder'])
        
    def set_train_dataset(self):
        self.train_dataset = load_dataset('glue', self.task, split='train')

    def set_val_dataset(self):
        self.val_dataset =  load_dataset('glue', self.task, split='val')

    def set_test_dataset(self):
         self.text_dataset = load_dataset('glue', self.task, split='test')
        
    def setup(self, stage: str):
        self.set_train_dataset()
        self.set_val_dataset()
        self.set_test_dataset()

    def train_dataloader(self):
        loader = DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            # num_workers=self.num_workers,
            # pin_memory=True,
            # collate_fn=self.collate_fn,
        )
        return loader

    def val_dataloader(self):
        loader = DataLoader(
            self.val_dataset,
            batch_size=self.eval_batch_size,
            shuffle=False,
            # num_workers=self.num_workers,
            # pin_memory=True,
            # collate_fn=self.collate_fn,
        )
        return loader
        
    def test_dataloader(self):
        loader = DataLoader(
            self.test_dataset,
            batch_size=self.eval_batch_size,
            shuffle=False,
            # num_workers=self.num_workers,
            # pin_memory=True,
            # collate_fn=self.collate_fn,
        )
        return loader

In [ ]:
config = copy.deepcopy(_config)
pl.seed_everything(_config["seed"])
model = METERTransformerSS(config)
model.current_tasks = ['mrpc']


dm = GlueDataModule(_config, 'mrpc', 32)

pl.seed_everything(_config["seed"])

exp_name = f'{_config["exp_name"]}'

os.makedirs(_config["log_dir"], exist_ok=True)
checkpoint_callback = pl.callbacks.ModelCheckpoint(
    save_top_k=1,
    verbose=True,
    monitor="val/the_metric",
    mode="max",
    save_last=True,
)
logger = pl.loggers.TensorBoardLogger(
    _config["log_dir"],
    name=f'{exp_name}_seed{_config["seed"]}_from_{_config["load_path"].split("/")[-1][:-5]}',
)

lr_callback = pl.callbacks.LearningRateMonitor(logging_interval="step")
callbacks = [checkpoint_callback, lr_callback]

num_gpus = (
    _config["num_gpus"]
    if isinstance(_config["num_gpus"], int)
    else len(_config["num_gpus"])
)

grad_steps = max(_config["batch_size"] // (
    _config["per_gpu_batchsize"] * num_gpus * _config["num_nodes"]
), 1)

max_steps = _config["max_steps"] if _config["max_steps"] is not None else None

trainer = pl.Trainer(
    devices=num_gpus,
    num_nodes=_config["num_nodes"],
    precision=_config["precision"],
    # accelerator="ddp",
    benchmark=True,
    deterministic=True,
    max_epochs=_config["max_epoch"] if max_steps is None else 1000,
    max_steps=max_steps,
    callbacks=callbacks,
    logger=logger,
    #prepare_data_per_node=False,
    #replace_sampler_ddp=False,
    accumulate_grad_batches=grad_steps,
    log_every_n_steps=10,
    # flush_logs_every_n_steps=10,
#     resume_from_checkpoint=_config["resume_from"],
    # weights_summary="top",
    fast_dev_run=_config["fast_dev_run"],
    val_check_interval=_config["val_check_interval"],
)

# log_dir = logger.log_dir
# eval_file = 'eval.txt'
# eval_path = os.path.join(log_dir, eval_file )
# setattr(model, f"eval_path", eval_path)
# f = open(eval_path,'w') 
# f.close()

if not _config["test_only"]:
    trainer.fit(model, datamodule=dm)
else:
    trainer.test(model, datamodule=dm)


## Import Calssification Head?

In [ ]:
model_type = model.text_encoder.base_model_prefix
model_name = model_type.capitalize()
model_type = 'bert'
model_name = model_type.capitalize()

In [ ]:
exec_string = f'from transformers.models.{model_type}.modeling_{model_type} import {model_name}ClassificationHead'

In [ ]:
from transformers.models.electra.modeling_electra import ElectraClassificationHead

In [ ]:
exec(exec_string)

## Image Encoder

In [ ]:
model.image_encoder(